# Лекция: Декораторы в Python

Для начала давайте вспомним, что такое области видимости в Python и какие они бывают?

In [ ]:
# LEGB

# Built-in
# Global
# Enclosing
# Local

def f():
    print(c)


c = 5
f()

5


<details>
<summary>
Почему мы смогли вывести значение переменной <b>с</b>?
</summary>
Потому что Python, не обнаружив переменную в локальной (local) области, начал искать переменную в следующих областях: enclosing -> global -> builtins. Здесь переменная <b>с</b> объявлена в глобальной области.
</details>

Теперь давайте посмотрим на другую функцию:

In [1]:
def lineq(*args):

    def get_x(x):
        return sum([args[i] * x**i for i in range(len(args))])

    return get_x

a = lineq(1, 2, 3)
print(a(1), a(2))
b = lineq(0, 0, 0)
print(b(100))

6 17
0


In [15]:
tes = lineq(1, 2, 3)
tes(0)

1

# Декораторы

В первую очередь, декоратор - это структурный паттерн проектирования. Входит в группу "Банды четырёх" - [link](https://www.geeksforgeeks.org/system-design/gang-of-four-gof-design-patterns/).

Ключевое преимущество: возможность добавлять новую логику к вызываемым объектам, не изменяя их исходный код.

Декоратор - это функция, которая принимает другую функцию и выполняет дополнительную логику.

На уровне языка декораторы появились в Python 2.5

In [6]:
# Самый базовый пример декоратора - функция для замера времени выполнения
import time
import datetime
import random

def say_hello(name: str) -> None:
    """Выводит строку вида 'Hello, <name>'"""
    time.sleep(random.randint(1, 3))
    print(f"Hello, {name}!")

start = datetime.datetime.now()
say_hello("Misha")
print(f"Time: {datetime.datetime.now() - start}")

Hello, Misha!
Time: 0:00:03.001986


**Почему бы не встроить замер времени сразу в say_hello? Ну или замерять как здесь - в месте, где вызываем?**

Обычно у функции есть конкретная задача/ответственность - замер времени перегружает код. Да и не все функции мы можем редактировать - многие функции импортируютс их сторонних библиотек. Что теперь, при каждом вызове писать вокруг код для работы со временем?

А если нам нужно замерять много разных функций? Везде будем тащить кучу импортов и писать одни и те же строки? Есть решение получше.

In [3]:
from typing import Any, Callable


def timed_func(func: Callable[..., Any]):
    def wrapped_func(*args, **kwargs) -> Any:
        start = datetime.datetime.now()
        res = func(*args, **kwargs)
        print(f"I am timed_func. Function {func.__name__} worked: {datetime.datetime.now() - start}")
        return res

    return wrapped_func

In [4]:
@timed_func
def say_hi(name: str) -> None:
    """Выводит строку вида 'Hi, <name>'"""
    time.sleep(random.randint(1, 3))
    print(f"Hi, {name}!")

In [7]:
say_hi("Tom Holland")

Hi, Tom Holland!
I am timed_func. Function say_hi worked: 0:00:03.001093


In [ ]:
timed_func(say_hi)("name")

Ай, красота! А помните нашу старую функцию say_hello? Давайте ей тоже время замерим!

In [34]:
timed_func(say_hello)("John Silver")

Hello, John Silver!
Time: 0:00:01.005429


Что здесь произошло? Давайте разберём в деталях:

```python
from typing import Any, Callable

def timed_func(func: Callable[..., Any]):
    def wrapped_func(*args, **kwargs):
        start = datetime.datetime.now()
        res = func(*args, **kwargs)
        print(f"Time: {datetime.datetime.now() - start}")
        return res

    return wrapped_func

timed_func(say_hello)("John Silver")
```

1. Мы вызвали функцию `timed_func`, передав ей целевую функцию `say_hello`
2. Функция `timed_func` вернула нам другую функцию, задача которой:
* принять аргументы
* запомнить время начала работы
* вызвать целевую функцию с ними
* замерить полное время работы
* вернуть результат целевой функции

3. Эту функцию мы вызываем с аргументом "John Silver".

Для того чтобы не записывать каждый раз такую цепочку функций, в Python есть синтаксический сахар:
```python
@timed_func
def say_hi(name: str) -> None:
    pass
```

Кстати, многие из вас подобное уже видели. Помните такое в 16 задании ЕГЭ?

In [ ]:
from functools import lru_cache

@lru_cache(12)
def F(n):
    if n==1: return 2
    if n>1: return 2*F(n-1)

for i in range(2, 1900):
    F(i)

print(F(1900)/2**1890)

1024.0


In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True, slots=True)
class Person:
    pass

Время посчитать, конечно, здорово. Балл за задачу получить - тоже. Но в реальности-то это хоть где-то используется?

**Конечно! Вот, например, PyTorch:**

```python
@torch.no_grad() # Отключает вычисление градиента.
def predict(model, input):
    output = model(input)
    return output
```

**А вот пример из FastAPI:**

```python
from fastapi import FastAPI

app = FastAPI()

@app.get("/")
def root():
    return {"message": "Hello mipt!"}
```

Да и куча других функций - для кэширования результатов, обработки транзакций, ролевого доступа и др.

Но мы ещё не всё посмотрели. Для следующего аспекта давайте посмотрим на атрибуты **\_\_doc\_\_**, **\_\_name\_\_** и **\_\_module\_\_**.

* **\_\_doc\_\_** позволяет обратиться к docstring объекта - полезно при вызове метода `help()`
* **\_\_name\_\_** хранит название объекта
* **\_\_module\_\_** хранит название модуля, где описан код нашего объекта


In [8]:
print(say_hello.__doc__)
print(say_hello.__name__)
print(say_hello.__module__)

Выводит строку вида 'Hello, <name>'
say_hello
__main__


In [9]:
print(say_hi.__doc__)
print(say_hi.__name__)
print(say_hi.__module__)

None
wrapped_func
__main__


Что тут будет?

<br>

In [ ]:
# Чиним через functools.wraps

from functools import wraps

def timed_func_fixed(func: Callable[..., Any]):

    @wraps(func)
    def wrapped_func(*args, **kwargs) -> Any:
        start = datetime.datetime.now()
        res = func(*args, **kwargs)
        print(f"I am timed_func_fixed. Function {func.__name__} worked: {datetime.datetime.now() - start}")
        return res

    return wrapped_func


@timed_func_fixed
def say_guten_tag(name: str) -> None:
    """Some doc"""
    time.sleep(random.randint(1, 3))
    print(f"Guten tag, {name}")


print(say_guten_tag.__doc__)
print(say_guten_tag.__name__)
print(say_guten_tag.__module__)

Some doc
say_guten_tag
__main__


In [54]:
# Кстати, декораторы можно комбинировать:

@timed_func
@timed_func_fixed
def say_guten_morgen(name: str) -> None:
    time.sleep(random.randint(1, 2))
    print(f"Guten morgen, {name}")

say_guten_morgen("Christophe")

Guten morgen, Christophe
I am timed_func_fixed. Function say_guten_morgen worked: 0:00:02.004211 seconds
I am timed_func. Function say_guten_morgen worked: 0:00:02.004250 seconds


Есть у декораторов ещё одна способность - передача аргументов для самого декоратора, а не декорируемой функции.

```python
from functools import lru_cache

@lru_cache(maxsize=10)
def F(n):
    pass
```

Всё очень просто! Просто добавляем ещё один уровень вложенности:

In [ ]:
def amortized_timed_func(call_count: int = 1):
    if call_count < 1:
        raise ValueError("Function must be called at least once!")
    def inner(func: Callable[..., Any]):
        def wrapper(*args, **kwargs) -> Any:
            cumulative_time = 0.0
            for _ in range(call_count):
                start = datetime.datetime.now()
                res = func(*args, **kwargs)
                cumulative_time += (datetime.datetime.now() - start).seconds

            print(f"I am amortized_timed_func. In average, function {func.__name__} worked: {cumulative_time/call_count} seconds")
            return res
        return wrapper
    return inner

In [70]:
@amortized_timed_func(5)
def sum_of_two(a: int, b: int) -> int:
    time.sleep(random.randint(0, 2))
    return a + b

sum_of_two(1, 3)

I am amortized_timed_func. In average, function sum_of_two worked: 0.6 seconds


4

И ещё один фокус декораторов:

In [18]:
calls_count = 0

def my_deco(func):
    def wrapped(*args, **kwargs):
        global calls_count
        calls_count += 1
        print(calls_count)
        return func(*args, **kwargs)
    return wrapped

@my_deco
def test_f():
    print()

@my_deco
def test_g():
    print

test_f()
test_f()
test_g()
test_f()

1

2

3
4



In [5]:
def count(f):
    total = 0
    print("total init!")

    def decorated(*args, **kwargs):
        nonlocal total
        total += 1
        return f(*args, **kwargs), total

    return decorated

In [ ]:
count(hello)("arg")

hello("arg")

In [8]:
@count
def hello(name):
    return f"Привет, {name}!"

@count
def bye(name):
    return f"Пока, {name}!"

print(hello("Пользователь_1"))
print(bye("Пользователь_1"))
print(hello("Пользователь_2"))
print(bye("Пользователь_2"))

total init!
total init!
('Привет, Пользователь_1!', 1)
('Пока, Пользователь_1!', 1)
('Привет, Пользователь_2!', 2)
('Пока, Пользователь_2!', 2)


In [ ]:
class Count:

    def __init__(self, func):
        self.func = func
        self.count = 0

    def __call__(self, *args, **kwargs):
        self.count += 1
        print(self.count)
        res = self.func(*args, **kwargs)
        return res

In [ ]:
class A:
    def __init__(self, a):
        self.a = a


abc = A("sd")
abc()

In [ ]:
@Count
def hello_class(name):
    return f"Привет, {name}!"

@Count
def bye_class(name):
    return f"Пока, {name}!"

print(hello_class("Пользователь_1"))
print(bye_class("Пользователь_1"))
print(hello_class("Пользователь_2"))
print(bye_class("Пользователь_2"))

Count(func)("Hello")

1
Привет, Пользователь_1!
1
Пока, Пользователь_1!
2
Привет, Пользователь_2!
2
Пока, Пользователь_2!


---

Внезапный вопрос к залу! Что такое **функтор** в Python?

<br>

# Пишем декоратор-функтор!

In [ ]:
import datetime
import random
import time


class TimedFunc:
    def __init__(self, some_arg):
        self.some_arg = some_arg

    def __call__(self, func):
        def wrapper(*args, **kwargs):
            start = datetime.datetime.now()
            res = func(*args, **kwargs)
            print(f"Time of {func.__name__}: {datetime.datetime.now() - start}")
            return res
        return wrapper


@TimedFunc(some_arg=5)
def say_bye(name) -> None:
    time.sleep(random.randint(1, 2))
    print(f"Bye, {name}")

say_bye("Andrew")

Bye, Andrew
Time of say_bye: 0:00:02.004109


# А что ещё бы такого задекорировать..?

In [25]:
import functools
import inspect
from typing import Any, Callable


def log_calls(cls: type) -> type:
    """
    Оборачивает публичные методы класса логированием входа/выхода.
    Пример упрощённый: не трогает свойства и dunder-методы.
    """
    for name, attr in list(vars(cls).items()):
        if name.startswith("_"):
            continue

        # Пропуск дескрипторов: property, classmethod, staticmethod - обрабатываются отдельно
        if isinstance(attr, property):
            continue

        if isinstance(attr, staticmethod):
            func = attr.__func__

            @functools.wraps(func)
            def wrapped(*args: Any, __f: Callable[..., Any] = func, **kwargs: Any) -> Any:
                print(f"call {cls.__name__}.{__f.__name__}()")
                return __f(*args, **kwargs)

            setattr(cls, name, staticmethod(wrapped))
            continue

        if isinstance(attr, classmethod):
            func = attr.__func__

            @functools.wraps(func)
            def wrapped(c: type, *args: Any, __f: Callable[..., Any] = func, **kwargs: Any) -> Any:
                print(f"call {cls.__name__}.{__f.__name__}()")
                return __f(c, *args, **kwargs)

            setattr(cls, name, classmethod(wrapped))
            continue

    return cls


@log_calls
class Calculator:
    def add(self, a: int, b: int) -> int:
        return a + b

    @staticmethod
    def mul(a: int, b: int) -> int:
        return a * b

    @classmethod
    def identity(cls) -> str:
        return cls.__name__

print(Calculator.mul(2, 4))

print(Calculator.identity())

call Calculator.mul()
8
call Calculator.identity()
Calculator


In [17]:
def log_method_call(func):
    def wrapper(self, *args, **kwargs):
        print(f"Вызов метода '{func.__name__}' с аргументами: {args}, {kwargs}")
        return func(self, *args, **kwargs)
    return wrapper

class MyClass:
    @log_method_call
    def do_something(self, value):
        return f"Сделано: {value}"

obj = MyClass()
obj.do_something(10)

Вызов метода 'do_something' с аргументами: (10,), {}


'Сделано: 10'

Потестим декораторы с наследованием:

In [29]:
def concat_prefix(func):
    def wrapper(self, *args, **kwargs):
        return "Prefix: " + func(self, *args, **kwargs)
    return wrapper


class A:
    @concat_prefix
    def say_hello(self, name):
        return f"Hello, {name}"


class B(A):
    ...


a = A()
a.say_hello("John")

b = B()
print(b.say_hello("Крош"))

Prefix: Hello, Крош


# Вопросы?